# README -> Workflow Extraction Pipeline
LangChain + LangGraph + OpenRouter (random model per call)

Pipeline: `repo_urls` (generator) -> LangGraph state machine (`scrape` -> `call_llm` -> `parse`) -> yields structured `WorkflowExtraction` objects.


To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio.
In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`.

## 1. Install dependencies

In [ ]:
!pip install -q langchain langchain-openai langgraph pydantic requests


In [ ]:
!pip install -q langchain-google-genai
import os

## 2. Config
Set your OpenRouter key (and optionally a GitHub token to avoid rate limits) below.
In Colab you can instead use `from google.colab import userdata` + `userdata.get('OPENROUTER_API_KEY')` if you've saved it as a Colab secret.

In [ ]:
# OpenRouter configuration removed as per user request.

In [ ]:
# Gemini configuration removed as per user request.

### Groq Configuration

To use Groq models, you'll need a Groq API key. Please add it to the Colab secrets manager under the "🔑" icon in the left panel, named `GROQ_API_KEY`. The following cell will then fetch it.

In [ ]:
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
# Pool of Groq model slugs
GROQ_MODEL_POOL = [
    "openai/gpt-oss-120b",   # Best general-purpose choice — strong reasoning, good for code/repo analysis, 131K context
    "openai/gpt-oss-20b",    # Faster & cheaper, still capable — good if you're processing many repos and want speed
    "groq/compound",         # If your pipeline needs live web search or code execution as part of the workflow
    "groq/compound-mini",    # Lighter version of the above, cheaper
]

In [ ]:
from google.colab import userdata

# Fetch GITHUB_TOKEN from Colab Secrets
github_token_from_secrets = userdata.get('GITHUB_TOKEN')
GITHUB_TOKEN = github_token_from_secrets if github_token_from_secrets is not None else ''

# You can also keep a direct paste option for testing, but ensure it's not committed
# GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "YOUR_GITHUB_TOKEN_HERE") # Replace with your actual token for direct testing

## 3. Imports

In [ ]:
import random
from typing import TypedDict, Optional, Iterator, List

import requests
from pydantic import BaseModel, Field

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langgraph.graph import StateGraph, START, END


In [ ]:
!pip install langchain_groq


In [ ]:
from langchain_groq import ChatGroq

## 4. Random LLM picker (OpenRouter)

In [ ]:
# get_random_openrouter_llm function removed as per user request.

In [ ]:
# get_random_gemini_llm function removed as per user request.

In [ ]:
def get_random_groq_llm() -> ChatGroq:
    """Return a ChatGroq client pointed at a randomly chosen Groq model."""
    model_name = random.choice(GROQ_MODEL_POOL)
    return ChatGroq(
        model_name=model_name,
        groq_api_key=GROQ_API_KEY,
        temperature=0.2,
    )

### How to use Groq models

To use Groq models in your pipeline, you'll need to modify the `llm_call_node` function in cell `960c3a5b` to call `get_random_groq_llm()` instead of `get_random_openrouter_llm()` (or `get_random_gemini_llm()`).

For example, change this line:
```python
    llm = get_random_openrouter_llm()
```
to this:
```python
    llm = get_random_groq_llm()
```
Remember to re-execute cell `960c3a5b` and then cell `NlgZhSWc2OxM` to recompile the graph after making this change.

## 5. Output schema + parser

In [ ]:
class WorkflowStep(BaseModel):
    step_number: int = Field(description="Order of this step in the workflow")
    action: str = Field(description="Short description of what happens in this step")
    command_or_code: Optional[str] = Field(
        default=None, description="Command, code snippet, or config, if any"
    )


class WorkflowExtraction(BaseModel):
    project_name: str = Field(description="Name of the project/repo, inferred from README")
    summary: str = Field(description="One or two sentence summary of what the project does")
    setup_steps: List[WorkflowStep] = Field(
        description="Ordered installation/setup steps extracted from the README"
    )
    usage_steps: List[WorkflowStep] = Field(
        description="Ordered usage/run steps extracted from the README"
    )


parser = PydanticOutputParser(pydantic_object=WorkflowExtraction)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You extract structured, step-by-step workflows from GitHub README files. "
            "Only use information present in the README. If a section is missing, "
            "return an empty list for it rather than inventing steps.\n\n"
            "{format_instructions}",
        ),
        ("human", "Repo URL: {repo_url}\n\nREADME content:\n{readme_text}"),
    ]
).partial(format_instructions=parser.get_format_instructions())


## 6. Scraper (GitHub raw README API)

In [ ]:
def scrape_readme(repo_url: str) -> Optional[str]:
    """Given https://github.com/owner/repo, fetch the raw README via the GitHub API."""
    try:
        owner_repo = repo_url.rstrip("/").split("github.com/")[-1]
        api_url = f"https://api.github.com/repos/{owner_repo}/readme"
        headers = {"Accept": "application/vnd.github.v3.raw"}
        if GITHUB_TOKEN:
            headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

        resp = requests.get(api_url, headers=headers, timeout=15)
        resp.raise_for_status()
        return resp.text
    except requests.RequestException as e:
        print(f"[scrape_readme] failed for {repo_url}: {e}")
        return None


## 7. LangGraph state + nodes

In [ ]:
class PipelineState(TypedDict, total=False):
    repo_url: str
    readme_text: Optional[str]
    model_used: Optional[str]
    raw_llm_output: Optional[str]
    workflow: Optional[WorkflowExtraction]
    error: Optional[str]


def scrape_node(state: PipelineState) -> PipelineState:
    readme = scrape_readme(state["repo_url"])
    if readme is None:
        return {**state, "error": "no README found or scrape failed"}
    return {**state, "readme_text": readme}


def parse_node(state: PipelineState) -> PipelineState:
    if state.get("error"):
        return state

    try:
        parsed = parser.parse(state["raw_llm_output"])
        return {**state, "workflow": parsed}
    except Exception as e:
        return {**state, "error": f"parse failed ({state.get('model_used')}): {e}"}

In [ ]:
def llm_call_node(state: PipelineState) -> PipelineState:
    if state.get("error"):
        return state

    # Using Groq LLMs exclusively as per user request.
    llm = get_random_groq_llm()

    chain = prompt | llm

    response = chain.invoke(
        {"repo_url": state["repo_url"], "readme_text": state["readme_text"]}
    )
    return {
        **state,
        "model_used": llm.model_name,
        "raw_llm_output": response.content,
    }

## 8. Build the graph

In [ ]:
builder = StateGraph(PipelineState)
builder.add_node("scrape", scrape_node)
builder.add_node("call_llm", llm_call_node)
builder.add_node("parse", parse_node)

builder.add_edge(START, "scrape")
builder.add_edge("scrape", "call_llm")
builder.add_edge("call_llm", "parse")
builder.add_edge("parse", END)

graph = builder.compile()

## 9. Outer generator: lazily drive the graph over many repo URLs

In [ ]:
def repo_url_stream(urls: List[str]) -> Iterator[str]:
    """Generator over repo URLs -- swap for a DB cursor, file reader, etc."""
    for url in urls:
        yield url


def run_pipeline(urls: List[str]) -> Iterator[PipelineState]:
    """Lazily process each repo URL through the graph, one at a time."""
    for url in repo_url_stream(urls):
        result_state = graph.invoke({"repo_url": url})
        yield result_state


## 10. Run it

In [ ]:
repo_urls = [

   ' https://github.com/prabindersinghh/opentelemetry-go-compile-instrumentation','https://github.com/prabindersinghh/khushsite','https://github.com/prabindersinghh/opensec-intelligence'
]

for result in run_pipeline(repo_urls):
    print("=" * 60)
    print("Repo:", result["repo_url"])
    if result.get("error"):
        print("Error:", result["error"])
        continue
    print("Model used:", result["model_used"])
    wf = result["workflow"]
    print("Project:", wf.project_name)
    print("Summary:", wf.summary)
    print("Setup steps:", len(wf.setup_steps))
    print("Usage steps:", len(wf.usage_steps))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')